In [8]:
import pandas as pd

### Tratar CSV de dados populacionais

In [9]:
# Carregar arquivo
df = pd.read_csv("../data/br_ibge_populacao_municipio.csv")

# Manter apenas municípios do RJ
# Códigos IBGE dos municípios do RJ começam com 33
df = df[df["id_municipio"].astype(str).str.startswith("33")]

# Manter apenas as colunas desejadas
df = df[["id_municipio", "ano", "populacao"]]

df = df[df["ano"] >= 2014]

# Renomear para ficar compatível com sua base
df = df.rename(columns={
    "id_municipio": "fmun_cod"
})

df["fmun_cod"] = df["fmun_cod"].astype("Int64")
df["populacao"] = df["populacao"].astype("Int64")

# Ordenar
df = df.sort_values(["ano", "fmun_cod"])

# Salvar
df.to_csv(
    "../data/populacao_rj.csv",
    index=False,
    encoding="utf-8"
)

print(df.head())
print(f"\nTotal de registros: {len(df)}")

        fmun_cod   ano  populacao
127433   3300100  2014     184940
127434   3300159  2014      10882
127435   3300209  2014     120948
127436   3300225  2014      11879
127437   3300233  2014      30439

Total de registros: 1104


In [10]:
### Tratar CSV de dados de segurança do RJ

In [11]:
# Carregar arquivo
df = pd.read_csv(
    "../data/BaseMunicipioMensal.csv",
    encoding="latin1",
    sep=";"
)

# Carregar mapeamento de regiões
regioes = pd.read_csv(
    "../data/municipios_rj_regioes_codigos.csv",
    encoding="utf-8-sig"
)

# Normalizar códigos para o merge
df["fmun_cod"] = df["fmun_cod"].astype(str).str.strip()
regioes["fmun_cod"] = regioes["fmun_cod"].astype(str).str.strip()

# Substituir a coluna regiao pelo valor correspondente do arquivo de mapeamento
regioes_map = regioes[["fmun_cod", "regiao"]].rename(columns={"regiao": "regiao_mapeada"})
df = df.merge(regioes_map, on="fmun_cod", how="left")
df["regiao"] = df["regiao_mapeada"].fillna(df["regiao"])
df = df.drop(columns=["regiao_mapeada"])

# Manter apenas anos até 2025
df = df[df["ano"] < 2026]

# Remover colunas desnecessárias
colunas_para_remover = [
    # Coluna temporal duplicada
    "mes_ano", 
    
    # Colunas que agregam outras colunas existentes
    "cvli", 
    "letalidade_violenta",
    "roubo_rua",
    "total_roubos",
    "total_furtos",
    
    # Colunas que não serão usadas
    "posse_drogas",
    "trafico_drogas",
    "apreensao_drogas_sem_autor",
    "recuperacao_veiculos",
    "apf",
    "aaapai",
    "cmp",
    "cmba",
    "ameaca",
    "pessoas_desaparecidas",
    "encontro_cadaver",
    "encontro_ossada",
    "pol_militares_mortos_serv",
    "pol_civis_mortos_serv",
    "registro_ocorrencias",
    "feminicidio",
    "tentativa_feminicidio",
    "fase"
]

df = df.drop(columns=colunas_para_remover)

# Salvar resultado
df.to_csv(
    "../data/BaseMunicipioMensal_2014_2025.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivo gerado com sucesso!")
print(df.head())
print(f"Registros: {len(df):,}")
print(f"Ano mínimo: {df['ano'].min()}")
print(f"Ano máximo: {df['ano'].max()}")

Arquivo gerado com sucesso!
  fmun_cod                fmun   ano  mes                 regiao  hom_doloso  \
0  3300100      Angra dos Reis  2014    1            Costa Verde          11   
1  3300159             Aperibé  2014    1    Noroeste Fluminense           0   
2  3300209            Araruama  2014    1    Baixadas Litorâneas           2   
3  3300225               Areal  2014    1  Centro-Sul Fluminense           0   
4  3300233  Armação dos Búzios  2014    1    Baixadas Litorâneas           2   

   lesao_corp_morte  latrocinio  hom_por_interv_policial  tentat_hom  ...  \
0                 0           0                        1           2  ...   
1                 0           0                        0           0  ...   
2                 0           0                        0           6  ...   
3                 0           0                        0           0  ...   
4                 0           0                        0           0  ...   

   furto_transeunte  furto_c